# Knowledge distillation tutorial
Links:
- https://www.philschmid.de/knowledge-distillation-bert-transformers 

# Task-specific distillation for BERT

This is end-to-end task-specific knowledge distilation Text-Classification example. Here we'll be using Transformers, PyTorch & Amazon SageMaker.

Distillation is the process of training a small "student" model to mimic a larger "teacher" model. 

In this example, we will use a [BERT-base](https://huggingface.co/textattack/bert-base-uncased-SST-2) as Teacher (~109 M) and [BERT-Tiny](https://huggingface.co/google/bert_uncased_L-2_H-128_A-2) (4.4 M) as Student. We will use [Text-Classification](https://huggingface.co/tasks/text-classification) as our task-specific knowledge distillation task and the [Stanford Sentiment Treebank v2 (SST-2)](https://huggingface.co/datasets/stanfordnlp/sst2) dataset for training.

They are two different types of knowledge distillation, the Task-agnostic knowledge distillation (right) and the Task-specific knowledge distillation (left). In this example we are going to use the Task-specific knowledge distillation.


In Task-specific knowledge distillation a "second step of distillation" is used to "fine-tune" the model on a given dataset. This idea comes from the [DistilBERT](https://huggingface.co/datasets/stanfordnlp/sst2) paper where it was shown that a student performed better than simply finetuning the distilled language model.

## Installation

In [1]:
%pip install transformers datasets tensorboard --upgrade
%pip install 'accelerate>=1.1.0'
# run in terminal:
# !sudo apt-get install git-lfs
# or
# brew install git-lfs
# git lfs install 


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup and configuration
In this example, we will use BERT-base as Teacher and BERT-Tiny as Student. Our Teacher is already fine-tuned on our dataset, which makes it easy for us to directly start the distillation training job rather than fine-tuning the teacher first to then distill it afterwards.

*IMPORTANT: This example will only work with a Teacher & Student combination where the Tokenizer is creating the same output.*



In [2]:
student_id = "google/bert_uncased_L-2_H-128_A-2"
teacher_id = "textattack/bert-base-uncased-SST-2"

In [3]:
from transformers import AutoTokenizer

/Users/ija/Desktop/small-LLM/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here we check, that teacher and student divide identical text into the same tokens. Otherwise it won'r work.

In [4]:
# init tokenizer
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
student_tokenizer = AutoTokenizer.from_pretrained(student_id)
 
# sample input
sample = "This is a basic example, with different words to test."
 
# assert results
assert teacher_tokenizer(sample) == student_tokenizer(sample), "Tokenizers haven't created the same output"
 

## Dataset and pre-processing

As Dataset we will use the [Stanford Sentiment Treebank v2 (SST-2)](https://huggingface.co/datasets/stanfordnlp/sst2) a text-classification for `sentiment-analysis`, which is included in the [GLUE benchmark](https://gluebenchmark.com/). The dataset is based on the dataset introduced by Pang and Lee (2005) and consists of 11,855 single sentences extracted from movie reviews. It was parsed with the Stanford parser and includes a total of 215,154 unique phrases from those parse trees, each annotated by 3 human judges. It uses the two-way (positive/negative) class split, with only sentence-level labels.

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer


In [6]:
dataset_id="nyu-mll/glue"
dataset_config="sst2"

In [7]:
dataset = load_dataset(dataset_id,dataset_config)

### Pre-processing & Tokenization

To distill our model we need to convert our "Natural Language" to token IDs.

We are going to use the tokenizer of the Teacher, though we could also go with the Student tokenizer.

In [8]:

tokenizer = AutoTokenizer.from_pretrained(teacher_id)

Additionally we add the `truncation=True` and `max_length=512` to align the length and truncate texts that are bigger than the maximum size allowed by the model.

In [9]:
def process(examples):
    tokenized_inputs = tokenizer(
        examples["sentence"], truncation=True, max_length=512
    )
    return tokenized_inputs

In [10]:
tokenized_datasets = dataset.map(process, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label","labels")
 
tokenized_datasets["test"].features

{'sentence': Value('string'),
 'labels': ClassLabel(names=['negative', 'positive']),
 'idx': Value('int32'),
 'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8'))}

`input_ids` -- stores tokenized sentences.

## Distilling the model using PyTorch and DistillationTrainer
Now that our `dataset` is processed, we can distill it. Normally, when fine-tuning a transformer model using PyTorch you should go with the `Trainer-API`. The [Trainer](https://huggingface.co/docs/transformers/v4.16.1/en/main_classes/trainer#transformers.Trainer) class provides an API for feature-complete training in PyTorch for most standard use cases.

In our example we cannot use the `Trainer` out-of-the-box, since we need to pass in two models, the `Teacher` and the `Student` and compute the loss for both. But we can subclass the `Trainer` to create a `DistillationTrainer` which will take care of it and only overwrite the [compute_loss](https://github.com/huggingface/transformers/blob/c4ad38e5ac69e6d96116f39df789a2369dd33c21/src/transformers/trainer.py#L1962) method as well as the `init` method. In addition to this we also need to subclass the `TrainingArguments` to include the our distillation hyperparameters.

In [11]:
from transformers import TrainingArguments, Trainer
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

W1022 17:19:48.656000 76773 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [22]:
class DistillationTrainingArguments(TrainingArguments):
    def __init__(self, *args, alpha=0.5, temperature=2.0, **kwargs):
        super().__init__(*args, **kwargs)
 
        self.alpha = alpha # to include two losses equally: 
                            # real label of sst and how much 
                            # distributions of student and teacher differ
        self.temperature = temperature # 2.0 by default 
                                        # logits are dividet by this value
                                        # student studies how confident was the teacher
        # because TrainingArguments does not have those args

class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model # adding teacher into the trainer
        # place teacher on same device as student
        self._move_model_to_device(self.teacher,self.model.device) # placing teacher on the same device 
                                                                    # so comparison of logits is valid
        self.teacher.eval() # turn on evaluation regime: no dropout, same logits for the same sentence;
                            # weights are not updated, it also does not get into the optimiser


    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
 
        # compute student output
        outputs_student = model(**inputs)
        student_loss=outputs_student.loss # cross-entropy loss
        # compute teacher output
        with torch.no_grad(): # weights are not being updated
          outputs_teacher = self.teacher(**inputs)
 
        # assert size
        assert outputs_student.logits.size() == outputs_teacher.logits.size()
 
        # Soften probabilities and compute distillation loss
        loss_function = nn.KLDivLoss(reduction="batchmean") 
        loss_logits = (loss_function(
            F.log_softmax(outputs_student.logits / self.args.temperature, dim=-1),
            F.softmax(outputs_teacher.logits / self.args.temperature, dim=-1)) * (self.args.temperature ** 2)) 
                                                                                # multiplication for scaling
        # Return weighted student loss
        loss = self.args.alpha * student_loss + (1. - self.args.alpha) * loss_logits
        return (loss, outputs_student) if return_outputs else loss

Kullback-Leibler loss (`KLDivLoss`):
- how much student's distributtion differs from teacher's
- teacher_p * log(teacher_p / student_p)

`reduction="batchmean"` averages those sums per batch:
$$\frac{1}{m}\sum^m_1(p_T(neg)\log(\frac{p_T(neg)}{p_S(neg)})+p_T(pos)\log(\frac{p_T(pos)}{p_S(pos)}))$$

### Hyperparameter Definition, Model Loading

In [28]:
# create label2id, id2label dicts for nice outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label
 
output_dir = "../models/tiny-bert-sst2"

# define training args
training_args = DistillationTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=7, # n of epoch for whole train-split
    per_device_train_batch_size=128, # how many sentences per step
    per_device_eval_batch_size=128,
    fp16=False, # True for GPU NVIDIA
    learning_rate=6e-5, # optimiser's step
    seed=33,

    # logging & evaluation strategies
    logging_strategy="epoch", # to get more information to TensorBoard
    eval_strategy="epoch", # accuracy for validation
    save_strategy="epoch", 
    save_total_limit=2, # n of last checkpoints
    load_best_model_at_end=True, 
    metric_for_best_model="accuracy",
    report_to="tensorboard",

    # distilation parameters
    alpha=0.5,
    temperature=4.0
    )
 
# define data_collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # padding fot too short sentences
 
# define model
teacher_model = AutoModelForSequenceClassification.from_pretrained(
    teacher_id,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
 
# define student model
student_model = AutoModelForSequenceClassification.from_pretrained(
    student_id,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

Loading weights: 100%|██████████| 39/39 [00:00<00:00, 5368.31it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing fro

### Evaluation metric

In [29]:
import numpy as np

In [30]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean().item()}

### Training

In [ ]:
trainer = DistillationTrainer(
    student_model,
    training_args,
    teacher_model=teacher_model,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [27]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 